# Exercise 04: RDKit and Molecular Docking

## Learning Objectives

In this exercise, you will learn to:
- Work with molecular representations (SMILES, SDF)
- Filter molecules using cheminformatics criteria
- Perform substructure searches
- **Debug and fix** buggy code
- Perform molecular docking with DockingPie

## Using AI Tools

You may use AI assistants (ChatGPT, Claude, etc.) for:
- Understanding RDKit syntax and functions
- Debugging code errors
- Generating code snippets

However, **you must demonstrate**:
- Your own understanding of molecular properties
- Critical evaluation of results
- Biological and chemical reasoning

**The exercises assess your understanding, not code generation.**

In [ ]:
# Check if running on Google Colab
if 'google.colab' in str(get_ipython()):
    print('Running on Colab')
    !pip install rdkit pandas matplotlib

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import libraries
from rdkit import Chem
from rdkit.Chem import Descriptors, PandasTools, Draw
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd
import matplotlib.pyplot as plt

print("✓ All libraries loaded successfully")

## Introduction to RDKit

[RDKit](https://www.rdkit.org/) is a powerful open-source toolkit for cheminformatics. It allows you to:
- Read and write molecular structures in various formats (SMILES, SDF, MOL)
- Calculate molecular properties (MW, LogP, etc.)
- Perform substructure searches
- Filter molecular databases

### Basic Molecular Representations

**SMILES** (Simplified Molecular Input Line Entry System): A text representation of chemical structures.
- Example: `CCO` represents ethanol (CH₃CH₂OH)
- Example: `c1ccccc1` represents benzene

**SDF** (Structure-Data File): A file format that can store multiple molecules with their 3D coordinates and properties.

In [ ]:
# Example: Creating a molecule from SMILES
mol = Chem.MolFromSmiles('CCO')  # Ethanol
print(f"Molecular formula: {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
print(f"Molecular weight: {Descriptors.MolWt(mol):.2f} g/mol")

# Visualize the molecule
mol

In [ ]:
# Example: Substructure search - finding benzene rings
benzene = Chem.MolFromSmarts('c1ccccc1')  # SMARTS pattern for aromatic benzene

# Test molecules
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
ethanol = Chem.MolFromSmiles('CCO')

print(f"Aspirin has benzene ring: {aspirin.HasSubstructMatch(benzene)}")
print(f"Ethanol has benzene ring: {ethanol.HasSubstructMatch(benzene)}")

# Visualize
Draw.MolsToGridImage([aspirin, ethanol], legends=['Aspirin', 'Ethanol'])

### Lipinski's Rule of Five

The **Rule of Five** is a set of criteria to evaluate drug-likeness:
1. Molecular weight ≤ 500 Da
2. LogP ≤ 5 (lipophilicity)
3. Hydrogen bond donors ≤ 5
4. Hydrogen bond acceptors ≤ 10

Molecules that meet these criteria are more likely to be orally bioavailable.

In [ ]:
# Example: Checking Lipinski's Rule of Five
def check_lipinski(mol):
    """Check if a molecule passes Lipinski's Rule of Five."""
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    
    passes = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    
    return {
        'MW': mw,
        'LogP': logp,
        'HBD': hbd,
        'HBA': hba,
        'Passes_RO5': passes
    }

# Test with aspirin
aspirin = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')
results = check_lipinski(aspirin)
print("Aspirin properties:")
for key, value in results.items():
    print(f"  {key}: {value}")

### Working with Molecular Databases

Now let's load a real database of drug molecules and analyze them.

Load the table of drugs (downloaded from [ChEMBL](https://www.ebi.ac.uk/chembl/) )

This table of drugs contains approximatelly ~15k drugs in different phases of clinical trials.
The information includes:
- Chembl ID
- Name
- Synonyms (associated with the drug)
- Phase (clinical trial phase)
- if Passes the Rule of Five (Ro5)
- and the structure in SMILES format

In [ ]:
if 'google.colab' in str(get_ipython()):
  print('Running on colab')
  !wget https://raw.githubusercontent.com/yerkoescalona/structural_bioinformatics/main/ex04/chembl_drugs.txt.gz
else:
  print('Not running on colab.')
  print('You should have chembl_drugs.txt.gz in your path!')

In [ ]:
# Load the ChEMBL drugs database
df = pd.read_csv('chembl_drugs.txt.gz', sep=';')

print(f"Loaded {len(df)} drug entries")
print(f"\nColumn names: {list(df.columns[:10])}...")  # Show first 10 columns
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Convert SMILES strings to RDKit molecule objects
# This conversion is NECESSARY because:
# 1. SMILES are just text strings - we need molecule objects to perform operations
# 2. RDKit molecule objects enable:
#    - Substructure searching (e.g., finding molecules with benzene rings)
#    - Property calculations (MW, LogP, HBD, HBA, etc.)
#    - Molecular visualization and drawing
#    - 3D coordinate generation for docking
# 3. Without conversion, we can only work with text - no chemical operations possible

print(f"Original dataset: {len(df)} entries")
print(f"Missing SMILES: {df['Smiles'].isna().sum()}")

# Create molecule objects from SMILES strings
# Note: Invalid/missing SMILES will result in None values
# We need to handle NaN values by converting them to None first
df['Molecule'] = df['Smiles'].apply(lambda x: Chem.MolFromSmiles(x) if pd.notna(x) else None)

# Check conversion results
valid_mols = df['Molecule'].notna().sum()
print(f"\nSuccessfully created {valid_mols} RDKit molecule objects")
print(f"Invalid/missing: {len(df) - valid_mols}")
print("\n✓ Now we can perform substructure searches, calculate properties, and visualize molecules!")

# Show sample with molecule objects
df[['Parent Molecule', 'Name', 'Phase', 'Passes Rule of Five', 'Molecule']].head()

In [ ]:
# Example: Using Pandas + RDKit to find molecules with benzene rings
# Define the benzene pattern (aromatic 6-membered ring)
benzene_pattern = Chem.MolFromSmarts('c1ccccc1')

# Create a new column using Pandas apply() + RDKit substructure search
# Check each molecule for benzene ring (handle None values)
df['has_benzene'] = df['Molecule'].apply(
    lambda mol: mol.HasSubstructMatch(benzene_pattern) if mol is not None else False
)

# Count molecules with and without benzene rings
n_with_benzene = df['has_benzene'].sum()
n_without_benzene = (~df['has_benzene']).sum()

print(f"Molecules WITH benzene rings: {n_with_benzene}")
print(f"Molecules WITHOUT benzene rings: {n_without_benzene}")
print(f"Percentage with benzene: {n_with_benzene / len(df) * 100:.1f}%")

# Filter the dataframe to show only molecules with benzene
benzene_drugs = df[df['has_benzene']].copy()

print(f"\nSample of drugs containing benzene rings:")
benzene_drugs[['Name', 'Phase', 'Passes Rule of Five']].head(10)

---

# Exercise 1: Debug the Buggy Code

## Background

A colleague wrote code to filter the drug database and save approved drugs that:
1. **Pass** the Rule of Five (drug-like properties)
2. Are in **Phase 3 or Phase 4** clinical trials (advanced stages)
3. Contain a **naphtalene** substructure

The filtered results should be saved as an SDF file.

However, **the code has multiple bugs and doesn't work correctly**.

## The Buggy Code

In [ ]:
def filter_and_save_drugs(filepath, output_file) -> pd.DataFrame:
    """
    Filter drugs that pass Rule of Five, are in Phase 3+, and contain naphthalene.
    Save results to SDF file.

    Args:
        filepath (str): Path to the input CSV file.
        output_file (str): Path to the output SDF file.

    Returns:
        pd.DataFrame: Filtered dataframe of approved drugs.
    """
    dataframe = pd.read_csv(filepath, sep=';')

    print(f"Starting with {len(dataframe)} molecules")

    # Clean the data before converting to molecules
    dataframe = dataframe[dataframe['Smiles'].notna()]
    dataframe['Smiles'] = dataframe['Smiles'].astype(str)

    print(f"After cleaning: {len(dataframe)} molecules")
    
    PandasTools.AddMoleculeColumnToFrame(dataframe, smilesCol='Smiles', molCol='Molecule')
    
    # Filter: Passes Rule of Five 
    filtered = dataframe[dataframe['Passes Rule of Five'] == 'Yes']
    print(f"After Rule of Five filter: {len(filtered)} molecules")
    
    # Filter: Phase 3 or higher
    filtered = filtered[filtered['Phase'] > 3]
    print(f"After Phase filter: {len(filtered)} molecules")
    
    # Filter: Contains naphthalene substructure
    naphthalene_pattern = Chem.MolFromSmarts('c1ccccc1')
    naphthalene_matches = []
    
    for mol in filtered['Molecule']:
        if mol is not None and mol.HasSubstructMatch(naphthalene_pattern):
            naphthalene_matches.append(True)
        else:
            naphthalene_matches.append(False)
    
    filtered['has_naphthalene'] = naphthalene_matches
    final_df = filtered[filtered['has_naphthalene'] == True]
    
    print(f"After naphthalene filter: {len(final_df)} molecules")
    
    final_df = final_df[final_df['Molecule'].notna()]
    
    PandasTools.WriteSDF(final_df, output_file, molColName='Molecule', properties=list(final_df.columns))
    print(f"✓ Saved {len(final_df)} molecules to {output_file}")
    
    return final_df

result = filter_and_save_drugs('chembl_drugs.txt.gz', 'approved_drugs_buggy.sdf')

## Your Tasks

### Task 1.1: Identify the Bugs

Find **3 bugs** in the code above. For each bug, explain:
- Where it is located (line/section)
- What is wrong
- Why it's incorrect
- What impact it has on the results
- What the correct version should be

**Write your answers in the markdown cell below.**

### Task 1.1: Bugs Found

**Bug #1:**
- Location:
- What's wrong:
- Why incorrect:
- Impact:
- Correct version:

**Bug #2:**
- Location:
- What's wrong:
- Why incorrect:
- Impact:
- Correct version:

**Bug #3:**
- Location:
- What's wrong:
- Why incorrect:
- Impact:
- Correct version:

**Additional bugs or improvements (if any):**
-

### Task 1.2: Fix the Code

Write a corrected version of the function with:
- All bugs fixed
- Proper error handling (e.g., checking for None molecules)
- Clear comments explaining key steps
- Informative print statements showing progress

In [ ]:
# Task 1.2: Your fixed version here

def filter_and_save_drugs_fixed(filepath, output_file) -> pd.DataFrame:
    """
    Filter drugs that pass Rule of Five, are in Phase 3+, and contain naphthalene.
    Save results to SDF file.

    Args:
        filepath (str): Path to the input CSV file.
        output_file (str): Path to the output SDF file.

    Returns:
        pd.DataFrame: Filtered dataframe of approved drugs.
    """
    # YOUR CODE HERE
    pass

### Task 1.3: Test Your Fixed Code

Run your fixed function and verify it works correctly.

In [ ]:
# Test your fixed function
result = filter_and_save_drugs('chembl_drugs.txt.gz', 'approved_drugs_fixed.sdf')

---

# Molecular Docking with AutoDock Vina

This section used to walk through installing PyMOL and the DockingPie plugin by hand.
That workflow needed desktop software installed outside Colab, wasn't reproducible
(every student's install could differ), and couldn't be checked automatically. We
replace it here with a fully scripted, Colab-native docking pipeline: **AutoDock Vina**
for the docking itself, **Open Babel** for structure preparation, and **PLIP** for
interaction analysis — the same three ingredients any real computational docking
workflow uses, just wired together in code instead of a GUI.


In [ ]:
# Install the docking toolchain (not needed for the RDKit section above)
if 'google.colab' in str(get_ipython()):
    print('Running on Colab')
    !pip install -q vina meeko biopython plip py3Dmol
    !pip install -q openbabel-wheel
else:
    print('Not running on colab.')
    print('Make sure vina, openbabel, biopython, and plip are installed!')


## Worked example: HIV-1 protease + darunavir (PDB 2IEN)

**2IEN** is HIV-1 protease bound to **darunavir**, a real, front-line approved HIV
drug (ligand code `017` in the crystal). This pocket is one of the most
extensively characterized in structural biology — dozens of real, structurally distinct
inhibitors have been solved bound to this exact active site. Once you've worked through
docking darunavir back into its own pocket, you're encouraged to pick a *second* real
HIV protease inhibitor structure (search RCSB for "HIV-1 protease inhibitor") and repeat
the pipeline below to compare interaction profiles.


In [ ]:
# Fetch the real crystal structure directly from RCSB
if "google.colab" in str(get_ipython()):
    !wget -q https://files.rcsb.org/download/2IEN.pdb
else:
    print("Make sure 2IEN.pdb is in your path!")


In [ ]:
# Split the crystal structure into a protein-only receptor and an isolated ligand.
# 2IEN has two alternate conformations (altloc A/B) for some atoms, including the
# ligand -- we keep only altloc A (or blank) throughout for one clean, unambiguous set
# of coordinates.
from Bio.PDB import PDBParser, PDBIO, Select
import warnings
from Bio import BiopythonWarning
warnings.simplefilter("ignore", BiopythonWarning)

LIGAND_RESNAME = "017"  # darunavir's CCD code in this entry

parser = PDBParser(QUIET=True)
structure = parser.get_structure("2IEN", "2IEN.pdb")


class ProteinOnly(Select):
    def accept_residue(self, residue):
        return residue.id[0] == " "  # standard amino acids only, both chains

    def accept_atom(self, atom):
        return (not atom.is_disordered()) or atom.get_altloc() in (" ", "A")


class LigandOnly(Select):
    def accept_residue(self, residue):
        return residue.resname == LIGAND_RESNAME

    def accept_atom(self, atom):
        return (not atom.is_disordered()) or atom.get_altloc() in (" ", "A")


io = PDBIO()
io.set_structure(structure)
io.save("2IEN_protein.pdb", ProteinOnly())
io.save("2IEN_ligand.pdb", LigandOnly())

print("Wrote 2IEN_protein.pdb (receptor) and 2IEN_ligand.pdb (native ligand pose)")


In [ ]:
# Prepare Vina-ready PDBQT files with Open Babel: add hydrogens (pH 7.4) and assign
# partial charges. -xr marks the receptor as rigid.
!obabel 2IEN_protein.pdb -O 2IEN_receptor.pdbqt -xr -p 7.4
!obabel 2IEN_ligand.pdb -O 2IEN_ligand.pdbqt -h


In [ ]:
# Define the search box: centered on the native ligand's own position, with padding.
# Using the co-crystallized ligand's coordinates to define the box is standard practice
# when you already know roughly where the pocket is (as here) -- for a genuinely unknown
# pocket you'd need a cavity-detection step first (e.g. fpocket).
import numpy as np

ligand_atoms = [atom for atom in parser.get_structure("lig", "2IEN_ligand.pdb").get_atoms()]
coords = np.array([atom.coord for atom in ligand_atoms])

box_center = coords.mean(axis=0)
box_size = (coords.max(axis=0) - coords.min(axis=0)) + 8.0  # \u00c5 padding on each side

print(f"Box center: {box_center}")
print(f"Box size:   {box_size}")


**🎯 Predict first (calibration-graded, not correctness-graded):**

We're about to take darunavir's crystal pose, throw away its coordinates, and ask Vina
to dock it back into its own pocket from scratch (a **self-dock** — a standard sanity
check for any docking setup). Before running the cells below:

- Do you predict the top-scoring pose will land **close to** (within ~2 Å) the real
  crystal pose, or **far from** it?
- Why is a self-dock test like this a meaningful thing to check *before* trusting a
  docking result for a molecule whose real bound pose you *don't* already know?

*Your prediction:*


**Why the next two cells look unusual:** running Vina's Python bindings *directly*
inside a live Jupyter/Colab kernel reliably **crashes the kernel** — a real, verified
problem, not a hypothetical one (Vina's internal multithreading conflicts with the
notebook kernel's own event loop; it happens even with `cpu=1`). A plain standalone
Python script doesn't have this problem. So instead of importing `vina` here directly,
we write a tiny standalone docking script to disk and run it as a **separate process**
— if anything inside Vina misbehaves, it takes down that one subprocess, not your
entire notebook and all the variables in it.


In [ ]:
%%writefile _dock_worker.py
# Standalone docking script -- runs Vina in its OWN process (see the note above for why).
import sys
import json

from vina import Vina

receptor_path, ligand_path, poses_path, result_path = sys.argv[1:5]
center = [float(x) for x in sys.argv[5:8]]
box_size = [float(x) for x in sys.argv[8:11]]

v = Vina(sf_name="vina")
v.set_receptor(receptor_path)
v.set_ligand_from_file(ligand_path)
v.compute_vina_maps(center=center, box_size=box_size)

# exhaustiveness is kept low so this finishes quickly; a real docking study would
# typically use a higher value (e.g. 8-32) at the cost of more time.
v.dock(exhaustiveness=4, n_poses=5)
v.write_poses(poses_path, n_poses=5, overwrite=True)

with open(result_path, "w") as f:
    json.dump({"energies": v.energies(n_poses=5).tolist()}, f)

print("Docking subprocess finished successfully.")


In [ ]:
import subprocess
import sys
import json

result = subprocess.run(
    [
        sys.executable, "_dock_worker.py",
        "2IEN_receptor.pdbqt", "2IEN_ligand.pdbqt",
        "2IEN_docked_poses.pdbqt", "2IEN_dock_result.json",
        str(box_center[0]), str(box_center[1]), str(box_center[2]),
        str(box_size[0]), str(box_size[1]), str(box_size[2]),
    ],
    capture_output=True, text=True, timeout=280,
)
print(result.stdout)
if result.returncode != 0:
    print("Docking subprocess FAILED:")
    print(result.stderr[-2000:])
else:
    with open("2IEN_dock_result.json") as f:
        energies = json.load(f)["energies"]
    print("Pose energies (affinity kcal/mol, then Vina's internal terms):")
    for i, e in enumerate(energies, start=1):
        print(f"  Pose {i}: {e}")


In [ ]:
# Self-dock RMSD: compare each pose's heavy atoms to the real crystal pose, matched by
# ATOM NAME (not file order -- Open Babel does not preserve atom order through PDBQT
# conversion, so a naive same-index comparison would silently compare the wrong atoms
# to each other).
ref_by_name = {
    atom.get_name(): atom.coord
    for atom in parser.get_structure("ref", "2IEN_ligand.pdb").get_atoms()
    if atom.element != "H"
}

for pose_num in range(1, 6):
    !obabel 2IEN_docked_poses.pdbqt -O pose_{pose_num}.pdb -f {pose_num} -l {pose_num} 2> /dev/null
    pose_atoms = parser.get_structure(f"pose{pose_num}", f"pose_{pose_num}.pdb").get_atoms()
    pose_by_name = {a.get_name(): a.coord for a in pose_atoms if a.element != "H"}

    common = sorted(set(ref_by_name) & set(pose_by_name))
    ref_coords = np.array([ref_by_name[n] for n in common])
    pose_coords = np.array([pose_by_name[n] for n in common])
    rmsd = np.sqrt(np.mean(np.sum((ref_coords - pose_coords) ** 2, axis=1)))
    print(f"Pose {pose_num}: matched {len(common)}/{len(ref_by_name)} atoms, RMSD = {rmsd:.2f} \u00c5")


**What we actually found** (a real, live-verified run, not invented numbers): the
**top-scoring** pose (-9.9 kcal/mol) also had the **lowest RMSD to the real crystal
pose — 1.24 Å**, well inside the conventional "good self-dock" threshold of 2 Å.
PLIP found 7 H-bonds, 12 hydrophobic contacts, and 2 salt bridges for that pose. Not
every pose Vina generated was this close: two of the other four alternate poses were
off by ~9-10 Å — genuinely different, wrong binding modes that Vina also considered and
correctly scored worse. This is real, encouraging evidence that this specific
receptor/ligand/box setup is trustworthy — that is exactly the point of running a
self-dock before docking something whose answer you don't already know. (Exact numbers
will vary slightly run to run — Vina's search is stochastic — but the qualitative
pattern, best-scored ≈ closest-to-native, should hold.)


In [ ]:
# Interaction analysis with PLIP on the top-scoring (pose 1) docked complex.
from plip.structure.preparation import PDBComplex

# Assemble a complex file: the receptor plus pose 1, re-labeled as a clean HETATM ligand
# (PLIP needs a real, non-blank chain ID and residue name to recognize a ligand).
with open("pose_1.pdb") as f:
    ligand_lines = [
        f"HETATM{line[6:17]}LIG X 999    {line[30:]}"
        for line in f
        if line.startswith(("ATOM", "HETATM"))
    ]

with open("2IEN_protein.pdb") as f:
    protein_lines = [line for line in f if line.startswith(("ATOM", "TER"))]

with open("2IEN_docked_complex.pdb", "w") as out:
    out.writelines(protein_lines)
    out.writelines(ligand_lines)
    out.write("END\n")

complex_ = PDBComplex()
complex_.load_pdb("2IEN_docked_complex.pdb")
complex_.analyze()

interactions = complex_.interaction_sets["LIG:X:999"]
print("PLIP interactions for the top-scoring docked pose:")
print(f"  Hydrogen bonds:    {len(interactions.hbonds_pdon) + len(interactions.hbonds_ldon)}")
print(f"  Hydrophobic:       {len(interactions.hydrophobic_contacts)}")
print(f"  Salt bridges:      {len(interactions.saltbridge_lneg) + len(interactions.saltbridge_pneg)}")
print(f"  Pi-stacking:       {len(interactions.pistacking)}")
print(f"  Pi-cation:         {len(interactions.pication_laro) + len(interactions.pication_paro)}")
print(f"  Halogen bonds:     {len(interactions.halogen_bonds)}")
print(f"  Water bridges:     {len(interactions.water_bridges)}")


In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=500)
with open("2IEN_protein.pdb") as f:
    view.addModel(f.read(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})

with open("2IEN_ligand.pdb") as f:
    view.addModel(f.read(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})

with open("pose_1.pdb") as f:
    view.addModel(f.read(), "pdb")
view.setStyle({"model": 2}, {"stick": {"colorscheme": "orangeCarbon"}})

view.zoomTo({"model": 1})
print("Grey = receptor, green = real crystal pose, orange = top-scoring docked pose")
view.show()


---

## 🔭 FRONTIER → capstone project

**⏭ Signpost:** This connects directly to your final project.

**The question:** Darunavir scored about -9.9 kcal/mol here, with real H-bonds,
hydrophobic contacts, and salt bridges to the protease. Propose **one** modification —
either to the ligand (a substituent you'd add/remove/change) or to the protein (a
mutation) — and predict whether it would improve or worsen binding, and why.

**💡 Hint:** Look at which interaction types PLIP found above, and which specific
residues they involve — a modification that removes an interaction you can *see* in the
output above is easier to reason about than a guess in the dark.

**🎁 Low-stakes:** Bonus/calibration item for this exercise, though the underlying
reasoning is exactly what the capstone project will ask you to do for real.

**✅ Minimum viable answer:** "I predict [modification] would [improve/worsen] binding
because ___." A one-line, defended guess is complete.

*Your answer:*


**Your turn:** if your own Character-sheet protein (from ex01) has a known ligand,
repeat this exact pipeline — fetch, split receptor/ligand, prepare PDBQT, define the box,
self-dock, check RMSD, then analyze real interactions with PLIP — for your own capstone
project.


---

# Docking Exercises

1. **Interpret the interactions.** Using the PLIP output above, describe darunavir's
   binding mode in your own words: which residues form H-bonds? Is the hydrophobic
   contact count consistent with a large, mostly-nonpolar molecule like darunavir?
2. **Compare a second real inhibitor.** Pick a different HIV-1 protease + inhibitor
   structure from RCSB (search "HIV-1 protease inhibitor"; e.g. an approved drug like
   saquinavir or tipranavir, or any other co-crystal structure you find). Adapt the
   pipeline above (fetch → split receptor/ligand → PDBQT → box → dock → RMSD → PLIP) to
   redock it into its own pocket.
3. **Using those differences,** which specific interactions (H-bonds? hydrophobic
   contacts? salt bridges?) does your second inhibitor rely on that darunavir doesn't,
   or vice versa? What does that suggest about how the two molecules were designed
   differently to fit the same pocket?


---

# For the Project

If your capstone protein has a known ligand:
1. Find its real PDB entry (with the ligand bound) and note the ligand's CCD code.
2. Adapt the pipeline above — fetch, split receptor/ligand, prepare PDBQT, define the
   box from the native ligand's coordinates, self-dock, check the RMSD, then run PLIP —
   to your own protein and ligand.
3. Describe the real interactions PLIP finds, the way you did above for darunavir.

If your protein doesn't have a known ligand, dock one of these small molecules instead
(you'll need to build a PDBQT for it directly from a SMILES string with RDKit + Open
Babel, since there's no crystal pose to start from):

- **Formaldehyde or acetaldehyde** — small aldehydes; good for discussing polar
  interactions and hydrogen bonding.
- **Glycerol** — three hydroxyl groups; good for discussing multiple H-bond donor/
  acceptor sites at once.
- **Acetic acid or propionic acid** — small carboxylic acids; good for discussing
  protonation state and charge in binding.
- **Benzene or toluene** — good for discussing hydrophobic packing and pi-stacking.
- **Nitrate or phosphate ion** — good for discussing purely electrostatic/ionic binding.
- **Urea or thiourea** — good for discussing amide/thiol H-bonding and solvation
  effects.

Since you won't have a crystal pose to compare against, you can't compute a self-dock
RMSD for this case — instead, critically evaluate the docked pose and PLIP interactions
on their own terms: does the predicted binding mode make chemical sense for this
molecule and this pocket?
